This file reads RHINO output data (pkl files) and writes them in an openPMD series  
Uses ADIOS2 as a backend (.bp5)  
We should use variable-based iteration encoding, but needs improvements  
For now we save all data at iteration zero  
Stores data in particle records (i.e. DataFrame-like)  
Data is stored in a shared folder on NERSC `/global/cfs/cdirs/m3239/2026_FES-AmSC/data/rhino`

In [ ]:
import openpmd_api as io
import numpy as np
import pandas as pd
from tqdm import tqdm
import sys
import networkx as nx

In [ ]:
# Configuration

OUTPUT_PATH = "/global/cfs/cdirs/m3239/aforment/IFE_AmSC/RHINO/scripts/tmp/rhino_particles.bp5"

RHINO_PATH = "/global/cfs/cdirs/m3239/2026_FES-AmSC/data/rhino/more_data_runs/" 
DATA_PATH  = f"{RHINO_PATH}/TBR/Data"
INPUT_PATH = f"{RHINO_PATH}/makeJSON.py"

sys.path.append(RHINO_PATH)
from makeJSON import InputFile

SECONDS_PER_DAY = 86400.0

In [ ]:
# Load RHINO Data: time-series and steady-state data
# Note 1: do the filenames include any information?
# Note 2: it would be nice if the structure were DATA_PATH/PREFIX/T.pkl, etc.
PREFIX="12-35-04"
INFIX ="IFE_AmSC_500MW_FuelCycle"
T_ts_df = pd.read_pickle(f"{DATA_PATH}/{PREFIX}_{INFIX}_T.pkl")
D_ts_df = pd.read_pickle(f"{DATA_PATH}/{PREFIX}_{INFIX}_D.pkl")
T_ss_df = pd.read_pickle(f"{DATA_PATH}/{PREFIX}_{INFIX}_T_SteadyState.pkl")
D_ss_df = pd.read_pickle(f"{DATA_PATH}/{PREFIX}_{INFIX}_D_SteadyState.pkl")
# Load RHINO Metafile
# Note: no infix in the metafile name 
meta_df = pd.read_pickle(f"{DATA_PATH}/{PREFIX}_IFE_meta.pkl")

In [ ]:
# Convert metadata to dictionary
meta = meta_df[0].to_dict()
for k, v in list(meta.items()):
    if isinstance(v, np.generic):
        meta[k] = v.item()

# We could also import directly as dictionary 
#import pickle 
#with open(f"{DATA_PATH}/{PREFIX}_IFE_meta.pkl", 'rb') as file:
#    meta = pickle.load(file)

# Get timestep
dt = float(meta["dt"]) if "dt" in meta else None
if dt is None:
    raise ValueError("dt is required for time-series data")   

endtime = float(meta["calc_length"])
times = np.arange(0, endtime+dt, dt)
nt = len(times)

In [ ]:
df_inputs = pd.DataFrame(InputFile['Systems_T']).T #
df_inputs.columns = ['subsystem_label', 
              'processing_time', 
              'nonradioactive_loss',
              'fractional_inflows', 
              'initial_mass',
              'source',
              'injectors', 
              'debug']

In [ ]:
labels_subsystems = df_inputs["subsystem_label"].to_list()
nSubsystems = len(labels_subsystems)
ids_subsystems = np.arange(nSubsystems, dtype=np.uint64)

In [ ]:
# Extract data from pkl files 
# Here we assume that T is defined in all the subsystems and D is defined in some of the subsystems
# Is this a safe assumption?

T_ts = np.ascontiguousarray(T_ts_df.to_numpy(dtype=np.float64))
nSubsystemsT, Nt = T_ts.shape
assert(nSubsystemsT==nSubsystems), "T_ts has the wrong number of subsystems"

D_ts = np.zeros((nSubsystems, Nt), dtype=np.float64)
for name, row in D_ts_df.iterrows():
    D_ts[df_inputs[df_inputs['subsystem_label']==name].index,:] = row.to_numpy(dtype=np.float64)

T_ss = T_ss_df.iloc[:, 0].to_numpy(dtype=np.float64)

D_ss = np.zeros((nSubsystems,), dtype=np.float64)
for name, row in D_ss_df.iterrows():
    D_ss[df_inputs[df_inputs['subsystem_label']==name].index] = float(row.iloc[0])

my_inventory = {}
my_inventory["Tritium"]   = {"data_ts": T_ts, "data_ss": T_ss}
my_inventory["Deuterium"] = {"data_ts": D_ts, "data_ss": D_ss}

In [ ]:
# Create Series
adios2_cfg = r'''
{
  "iteration_encoding": "variable_based",
  "adios2": {
    "modifiable_attributes": false,
    "use_group_table": false,
    "engine": {
      "type": "bp5",
      "parameters": {
        "StatsLevel": "0",
        "AsyncWrite": "guided"
      }
    }
  }
}
'''
OUTPUT_PATH = "/global/cfs/cdirs/m3239/aforment/IFE_AmSC/RHINO/scripts/tmp/rhino_particles.bp5"
series = io.Series(OUTPUT_PATH, io.Access_Type.create_linear, adios2_cfg)
print("Writing RHINO data in particle representation...")

### Ideal attributes

In [ ]:
# =================
# Software Metadata
series.set_attribute("software/softwareName", "RHINO")
series.set_attribute("software/softwareDescription", "RHINO: Fusion Pilot Plant fuel cycle simulation")
series.set_attribute("software/softwareVersion", "1.0")
# Placeholder for additional attributes under software/
series.set_attribute("software/versionControlSoftware", "")
series.set_attribute("software/softwareCommit", "")
series.set_attribute("software/softwareDocumentation", "")

In [ ]:
# ===================
# Provenance metadata
series.set_attribute("provenance/author", "Holly Flynn")
series.set_attribute("provenance/authorAffiliation", "Savannah River National Laboratory")
series.set_attribute("provenance/authorEmail", "Holly.Flynn@srnl.doe.gov")
series.set_attribute("provenance/creationDate", "2025-12-16")
# Placeholder for additional attributes under provenance/
series.set_attribute("provenance/inputDirectory", "")
series.set_attribute("provenance/inputFiles", "")
series.set_attribute("provenance/originalDataDirectory", "")
series.set_attribute("provenance/originalDataFiles", "")

In [ ]:
# ==============================
# System metadata (placeholders)
series.set_attribute("system/systemIP", "")
series.set_attribute("system/systemDescription", "")

In [ ]:
# =====================================
# Application metadata (RHINO-specific)
series.set_attribute("metadata/subsystems/description", "Subsystems of the pilot plant fuel cycle")
series.set_attribute("metadata/subsystems/id", ids_subsystems.tolist())
series.set_attribute("metadata/subsystems/labels", labels_subsystems)
# series.set_attribute("metadata/subsystems/mapping", canon)        # dict as an attribute is not supported
# Placeholder for additional attributes under metadata/subsystems/
series.set_attribute("metadata/subsystems/connections", "")
series.set_attribute("metadata/subsystems/connectionType", "")
series.set_attribute("metadata/species/description", "Gas species in the fuel cycle")
series.set_attribute("metadata/species/names", ["Tritium", "Deuterium"])
# Placeholder for additional attributes under metadata/species/
series.set_attribute("metadata/species/Tritium/subsystems", "")
series.set_attribute("metadata/species/Tritium/subsystemsConnection", "")
series.set_attribute("metadata/species/Deuterium/subsystems", "")
series.set_attribute("metadata/species/Deuterium/subsystemsConnection", "")

### Series attributes today 

In [ ]:
series.particles_path = "inventory"
series.set_attribute("software", "RHINO")
series.set_attribute("softwareVersion", "1.0")
series.set_attribute("softwareDescription", "RHINO: Fusion Pilot Plant fuel cycle simulation")
series.set_attribute("author", "Holly Flynn")
series.set_attribute("authorAffiliation", "Savannah River National Laboratory")
series.set_attribute("authorEmail", "Holly.Flynn@srnl.doe.gov")

### Iteration(s)

In [ ]:
it = series.snapshots()[0]
it.time = 0.0
it.dt = float(dt)
it.time_unit_SI = SECONDS_PER_DAY

### Species

In [ ]:
# it.particles["electrons"]["momentum"]["x"]

In [ ]:
# subsystems
species = it.particles["subsystems"]
species.set_attribute("labels", labels_subsystems)
species.set_attribute("description", "Subsystems of the pilot plant fuel cycle")

record = species["id"]

data = np.ascontiguousarray(ids_subsystems).copy()
dataset = io.Dataset(ids_subsystems.dtype, ids_subsystems.shape)

component = record[io.Record_Component.SCALAR]
component.reset_dataset(dataset)
component.store_chunk(data)

In [ ]:
# times
species = it.particles["tmp"] # i don't know how to call this one 
species.set_attribute("description", "Times")

record = species["times"]
record.unit_dimension =  {io.Unit_Dimension.T: 1}
record.unit_SI = SECONDS_PER_DAY

data = np.ascontiguousarray(times).copy()
dataset = io.Dataset(times.dtype, times.shape)

component = record[io.Record_Component.SCALAR]
component = record[io.Record_Component.SCALAR]
component.reset_dataset(dataset)
component.store_chunk(data)


In [ ]:
def write_species(name, data_ts, data_ss):

    pt = it.particles[name]

    pt.set_attribute("description", "Inventory across subsystems for species " + name)
    pt.set_attribute("timeAxis", 1)  
    pt.set_attribute("subsystemsAxis", 0)

    # time-series inventory data (copy to ensure writable for store_chunk)
    data_arr = np.ascontiguousarray(data_ts).copy()
    inv_rec = pt["mass"][io.Record_Component.SCALAR]
    inv_rec.reset_dataset(io.Dataset(data_arr.dtype, data_arr.shape))
    inv_rec.store_chunk(data_arr)

    pt["mass"].unit_dimension = {io.Unit_Dimension.M: 1}
    inv_rec.unit_SI = 1e-3
    
    # steady-state inventory data (copy to ensure writable for store_chunk)
    ss_arr = np.ascontiguousarray(data_ss).copy()
    ss_rec = pt["mass_steady"][io.Record_Component.SCALAR]
    ss_rec.reset_dataset(io.Dataset(ss_arr.dtype, ss_arr.shape))
    ss_rec.store_chunk(ss_arr)

    pt["mass_steady"].unit_dimension = {io.Unit_Dimension.M: 1}
    ss_rec.unit_SI = 1e-3

    

write_species("Tritium", T_ts, T_ss)
write_species("Deuterium", D_ts, D_ss)

#series.flush()

it.close()
series.close()

print("RHINO data written to ADIOS-OpenPMD in particle representation.")
print("Output:", OUTPUT_PATH)